In [ ]:
import pickle
from pathlib import Path

import numpy as np
import soundfile as sf
from tqdm import tqdm


In [ ]:
def fmt_param(v: float) -> str:
    s = f"{float(v):.2f}".rstrip("0").rstrip(".")
    return s if s else "0"


def write_wav(path: Path, audio: np.ndarray, sr: int):
    path.parent.mkdir(parents=True, exist_ok=True)
    x = np.asarray(audio, dtype=np.float32)
    if x.ndim == 2 and x.shape[0] < x.shape[1]:
        x = x.T
    x = np.clip(x, -1.0, 1.0)
    sf.write(str(path), x, sr, subtype="PCM_16")


def equally_spaced_indices(n: int, frac: float) -> np.ndarray:
    k = max(1, int(round(n * frac)))
    return np.unique(np.linspace(0, n - 1, k, dtype=int))


In [ ]:
def save_cl1b_pickle_to_wav(
    pickle_path: str,
    inputs_out_dir: str,
    outputs_out_dir: str,
    sr: int = 48000,
):
    pkl = Path(pickle_path)
    in_dir = Path(inputs_out_dir)
    out_dir = Path(outputs_out_dir)

    with open(pkl, "rb") as f:
        data = pickle.load(f)

    x = data.get("x", None)
    y = data.get("y", None)
    z = data.get("z", None)

    n_candidates = []
    for arr in (x, y, z):
        if isinstance(arr, np.ndarray):
            n_candidates.append(arr.shape[0])
    if not n_candidates:
        raise RuntimeError("No arrays 'x'/'y'/'z' found in the pickle.")

    n = n_candidates[0]
    if not all(m == n for m in n_candidates):
        shapes = {k: v.shape for k, v in (("x", x), ("y", y), ("z", z)) if isinstance(v, np.ndarray)}
        raise RuntimeError(f"Mismatch in example counts: {shapes}")

    print(f"Loaded: {pkl.name}  |  N={n}  |  sr={sr}")
    if x is not None:
        print(f"Writing inputs -> {in_dir}")
        for i in tqdm(range(n), desc="inputs"):
            write_wav(in_dir / f"input_{i + 1}.wav", x[i], sr)

    if y is not None:
        print(f"Writing outputs -> {out_dir}")
        for i in tqdm(range(n), desc="outputs"):
            if z is not None and z.shape[1] >= 4:
                th, rt, at, rl = z[i, 0], z[i, 1], z[i, 2], z[i, 3]
                name = (
                    f"output_{i + 1}"
                    f"_th_{fmt_param(th)}"
                    f"_rt_{fmt_param(rt)}"
                    f"_at_{fmt_param(at)}"
                    f"_rl_{fmt_param(rl)}.wav"
                )
            else:
                name = f"output_{i + 1}.wav"
            write_wav(out_dir / name, y[i], sr)


In [ ]:
def save_cl1b_trainval_to_wav(
    train_pickle_path: str,
    base_out_dir: str,
    sr: int = 48000,
    val_frac: float = 0.10,
    preserve_indices: bool = True,
):
    base = Path(base_out_dir)
    tr_in = base / "train" / "inputs"
    tr_out = base / "train" / "outputs"
    va_in = base / "val" / "inputs"
    va_out = base / "val" / "outputs"

    with open(train_pickle_path, "rb") as f:
        data = pickle.load(f)

    x = data.get("x", None)
    y = data.get("y", None)
    z = data.get("z", None)

    if not isinstance(x, np.ndarray) or not isinstance(y, np.ndarray):
        raise RuntimeError("Expected 'x' and 'y' arrays in the train pickle.")
    if not isinstance(z, np.ndarray) or z.shape[1] < 4:
        raise RuntimeError("Expected 'z' with shape (N, 4).")

    n = x.shape[0]
    if y.shape[0] != n or z.shape[0] != n:
        raise RuntimeError(f"Mismatched N: x={x.shape}, y={y.shape}, z={z.shape}")

    val_idx = equally_spaced_indices(n, val_frac)
    val_mask = np.zeros(n, dtype=bool)
    val_mask[val_idx] = True
    train_idx = np.where(~val_mask)[0]

    print(f"Loaded {Path(train_pickle_path).name}: N={n}, sr={sr}")
    print(f"Validation size: {len(val_idx)}")
    print(f"Train size: {len(train_idx)}")

    def fname_index(i: int, k: int) -> int:
        return i + 1 if preserve_indices else k + 1

    print(f"Writing train to {tr_in.parent}")
    for k, i in enumerate(tqdm(train_idx, desc="train")):
        idx_for_name = fname_index(i, k)
        write_wav(tr_in / f"input_{idx_for_name}.wav", x[i], sr)
        th, rt, at, rl = z[i, 0], z[i, 1], z[i, 2], z[i, 3]
        out_name = (
            f"output_{idx_for_name}"
            f"_th_{fmt_param(th)}"
            f"_rt_{fmt_param(rt)}"
            f"_at_{fmt_param(at)}"
            f"_rl_{fmt_param(rl)}.wav"
        )
        write_wav(tr_out / out_name, y[i], sr)

    print(f"Writing val to {va_in.parent}")
    for k, i in enumerate(tqdm(val_idx, desc="val")):
        idx_for_name = fname_index(i, k)
        write_wav(va_in / f"input_{idx_for_name}.wav", x[i], sr)
        th, rt, at, rl = z[i, 0], z[i, 1], z[i, 2], z[i, 3]
        out_name = (
            f"output_{idx_for_name}"
            f"_th_{fmt_param(th)}"
            f"_rt_{fmt_param(rt)}"
            f"_at_{fmt_param(at)}"
            f"_rl_{fmt_param(rl)}.wav"
        )
        write_wav(va_out / out_name, y[i], sr)


In [ ]:
test_path = "CL1B_analog_test.pickle"
train_path = "CL1B_analog_train.pickle"

test_inputs_out = "/CL1B/test/inputs"
test_outputs_out = "/CL1B/test/outputs"

train_inputs_out = "/CL1B/train/inputs"
train_outputs_out = "/CL1B/train/outputs"

save_cl1b_pickle_to_wav(test_path, test_inputs_out, test_outputs_out, sr=48000)
save_cl1b_pickle_to_wav(train_path, train_inputs_out, train_outputs_out, sr=48000)

base_out = "output_path"
save_cl1b_trainval_to_wav(
    train_pickle_path=train_path,
    base_out_dir=base_out,
    sr=48000,
    val_frac=0.10,
    preserve_indices=True,
)
